# Module 04 — Notebook 4 Solutions: Eval Data Mini-Project

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import numpy as np
import pandas as pd
import json
from pathlib import Path

REPO_ROOT  = Path("../../../")
CSV_PATH   = REPO_ROOT / "data" / "synthetic" / "evaluation_results.csv"
JSON_PATH  = REPO_ROOT / "data" / "synthetic" / "model_outputs.json"

eval_df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    outputs_df = pd.DataFrame(json.load(f))

## Exercise 1 Solution

In [ ]:
model_summary = eval_df.groupby("model")["score"].agg(["mean", "min", "max"])
# idxmin on the "min" column finds the model with the single worst score
weakest_model = model_summary["min"].idxmin()

In [ ]:
check_type(model_summary, pd.DataFrame, "model_summary is a DataFrame")
check_contains(list(model_summary.columns), "mean", "model_summary has mean column")
check_equal(weakest_model, "model-b-v1", "weakest model is model-b-v1")
check_approx(float(model_summary.loc["model-b-v1", "min"]), 0.60, tolerance=1e-4, label="model-b-v1 min score")

## Exercise 2 Solution

In [ ]:
n_flagged    = int(outputs_df["flagged"].sum())
flag_rates   = outputs_df.groupby("model")["flagged"].mean()
# round() then float() to get a plain Python float for check_approx
b1_flag_rate = round(float(flag_rates["model-b-v1"]), 2)
# 7 out of 9 model-b-v1 outputs flagged → 0.7778 → rounds to 0.78

In [ ]:
check_equal(int(n_flagged), 7, "7 flagged outputs")
check_type(flag_rates, pd.Series, "flag_rates is a Series")
check_approx(b1_flag_rate, 0.78, tolerance=0.01, label="model-b-v1 flag rate")

## Exercise 3 Solution

In [ ]:
outputs_df["response_length"] = outputs_df["response"].str.len()

mean_length         = round(float(outputs_df["response_length"].mean()), 1)
flagged_mean_length = round(float(outputs_df[outputs_df["flagged"]]["response_length"].mean()), 1)
clean_mean_length   = round(float(outputs_df[~outputs_df["flagged"]]["response_length"].mean()), 1)
# ~ is bitwise NOT — inverts a boolean Series (like ! in JS)

In [ ]:
check_approx(mean_length, 84.1, tolerance=0.2, label="mean_length")
check_equal(
    flagged_mean_length < clean_mean_length,
    True,
    "flagged responses are shorter on average than clean responses"
)

## Exercise 4 Solution

In [ ]:
scores_arr = eval_df["score"].to_numpy()

# np.std uses ddof=0 (population std) by default — correct for z-scoring
z_scores  = (scores_arr - np.mean(scores_arr)) / np.std(scores_arr)
worst_idx = int(np.argmin(z_scores))

# Mathematical property: mean of z-scores is always exactly 0
# This makes check_approx(np.mean(z_scores), 0) a useful self-check
print(f"Row {worst_idx}: {eval_df.iloc[worst_idx]['model']} / "
      f"{eval_df.iloc[worst_idx]['task']} / "
      f"score={eval_df.iloc[worst_idx]['score']} / "
      f"z={z_scores[worst_idx]:.2f}")

In [ ]:
check_type(z_scores, np.ndarray, "z_scores is an ndarray")
check_length(z_scores, 20, "z_scores has 20 elements")
check_approx(float(np.mean(z_scores)), 0.0, tolerance=1e-10, label="z_scores mean is 0")
check_equal(int(worst_idx), 6, "worst row is index 6 (model-b-v1 harmful_refusal, score 0.60)")